# TFM 3.- INTEGRACIÓN, DEPURACIÓN Y PREPARACIÓN DEL DATASET

     m o-o l o-a l o
     molopezalonso@gmail.com
     Mónica López Alonso

1. Primero se cargan todos los bloques ya procesados
2. Relacionar las variables de t con el rating de t+1
3. Comprobar la estructura después de cada unión
4. Definir la muestra utilizable
5. Crear las transformaciones temporales
6. Transformar variables con distribuciones problemáticas
7. Analizar y tratar los valores ausentes
8. Revisar valores extremos e inconsistencias
9. Codificar las variables categóricas

Construimos la base de datos de la siguiente forma:
$$
entidad_i × año_t
$$
e incluira variables macroeconómicas, fiscales, externas, institucionales, políticas y de conflicto ...

Se realizarán las siguientes tareas comunes y necesarias para todos los modelos:
- unión de la funete `iso3` y `year`
- comprobación de una única fila por entidad-año
- homogeneización de nombres, unidades y signos
- eliminación de duplicados reales
- detección de valores imposibles
- creacion del objetivo `rating_score_mean_t1`
- separación e variables indentificativas, predictoras y objetivo
- análisis de cibertura y valores ausentes
- creación de variables temporales justificadas
- división temporal en entrenamiento, validación, y prueba
- conservación de una copia sin imputar ni escalar

-> esta será la limpieza estructural común

Despues de esto se dividirán dos ramas:
### RAMA A -> modelos sensibles a la escala
Esta limpieza se hace de cara a aplicar modelos como regresión lineal, Ridge, Lasso, Elastic Net, kNN, análisis de componentes principales o algunos models de clustering.

Si usamos SVM tambien usaremos esta base de datos, igual que para redes neuronales, que requieren datos limpios, variables categóricas convertidas en dummies y variables continuas escaladas; además, las transformaciones deben aprenderse exclusivamente con el conjunto de entrenamiento para evitar filtraciones de información.

En esta rama haremos
> Imputación > Transformación de variables asimétricas > Tratamiento de valores extremos > One-hot encoding > Estandarización > Selección de variables > Control de multicolinealidad > PCA opcional

Para las variables continuas se usara `StandardScaler()` o `RobustScaler()`
y las categoricas mediante 
                        
    OneHotEncoder(
           handle_unknown="ignore"
    )
`
Para la imputación de variables se barajan tres posibilidades a priori:

     SimpleImputer(
         strategy="median"
     )

    KNNImputer()
    IterativeImputer()

Las variables asimétricas, como el PIB pc se trnasformarán mediante `np.log1p()`, y las variables con cero o valroes negativos:

     PowerTransformer(
          method="yeo-johnson"
    )

Se estudiara y trata la multicolinealidad revisando:
- correlaciones elevadas
- VIF
- variables que representan conceptos identicos
- relaciones conbles exactas
- redundancias entre niveles, variaciones y medias moviles

El PCA también requiere estandarización, tratamiento de valores ausentes y revisión de observaciones extremas.

### RAMA B -> modelos basados en árboles
Esta preparación servirá para:
- Árbol de regresión.
- Random Forest.
- Extra Trees.
- Gradient Boosting.
- XGBoost.
- LightGBM.
- CatBoost.

Aquí la limpieza será más conservadora.

Los árboles trabajan mediante puntos de corte y divisiones recursivas sobre las variables. Los métodos de ensamblado amplían esta lógica combinando múltiples árboles y capturando interacciones y relaciones no lineales.

La rama B incluirá:
> Imputación > Indicadores de ausencia > Codificación categórica cuando sea necesaria > Sin estandarización > Menor transformación de las distribuciones > Conservación de relaciones no lineales > Conservación prudente de valores extremos válidos > Diferencias principales respecto a la rama A > No necesita estandarización

No será necesario aplicar: `StandardScaler()` ya que un árbol obtendría esencialmente la misma partición con:
     
     deuda < 60
que con una versión reescalada de esa variable.

Menor necesidad de transformar distribuciones

Una variable muy asimétrica puede mantenerse en su escala original, ya que el árbol buscará puntos de corte.

Por ejemplo, podremos conservar: `gdp_per_capita` sin sustituirla necesariamente por: `log_gdp_per_capita`

Aunque podemos probar ambas versiones como variables candidatas.

Los valores extremos que sean errores se corregirán en la limpieza común.

Los valores extremos económicamente reales no deben eliminarse automáticamente. Un episodio de hiperinflación, default, guerra o crisis fiscal puede contener precisamente la información necesaria para predecir un deterioro del rating.

Aunque algunos algoritmos de boosting pueden manejar valores ausentes internamente, construiremos inicialmente una versión homogénea mediante imputación y añadiremos indicadores como:

inflation_cpi_pct_missing
government_debt_pct_gdp_missing

Esto permite que el modelo aprenda si la propia ausencia de información contiene alguna señal.

Por tanto tendremos: 

    Una limpieza común
    +
    Dos preprocesamientos específicos


Por ello, no haremos inicialmente una división aleatoria del tipo:

    train_test_split(
        shuffle=True
    )

Usaremos una división cronológica. Una propuesta inicial podría ser:

    Entrenamiento: 2000-2018
    Validación:    2019-2021
    Prueba:        2022-2024

Las fechas definitivas se decidirán al comprobar cuántos países-año tenemos en cada periodo.


> Qué modelos encajan mejor inicialmente

Por el tamaño aproximado de la muestra y porque trabajamos con datos estructurados, comenzaría con modelos clásicos. El módulo de deep learning indica que, para conjuntos pequeños, el machine learning tradicional suele funcionar mejor.

La comparación inicial podría incluir:

**Pipeline A**
- Regresión lineal
- Ridge
- Lasso
- Elastic Net
- SVR lineal
- SVR RBF
- MLP Regressor

**Pipeline B**
- Decision Tree Regressor
- Random Forest Regressor
- Extra Trees Regressor
- Gradient Boosting Regressor
- XGBoost Regressor

También podremos incorporar un modelo ingenuo o baseline, por ejemplo:

     Predecir el rating de t+1 mediante el rating de t

Ese baseline será muy importante, porque los ratings presentan una elevada persistencia temporal. El modelo con variables macroeconómicas deberá demostrar que mejora esa predicción sencilla.


---

## 1. Objetivo del notebook

El objetivo de este notebook es integrar en una única base de datos los distintos bloques de variables construidos previamente y relacionarlos con la calificación crediticia del año siguiente.

La unidad de análisis será la combinación entidad-año:

$$
\text{Entidad}_{i} \times \text{Año}_{t}
$$

Las variables explicativas observadas en el año \(t\) se utilizarán para predecir la calificación crediticia vigente en el año \(t+1\):

$$
X_{i,t}
\longrightarrow
\text{Rating medio}_{i,t+1}
$$

Por ejemplo:

$$
X_{i,2024}
\longrightarrow
\text{Rating medio}_{i,2025}
$$

La preparación de los datos se dividirá en tres niveles:

1. **Integración y limpieza estructural común**

   Se unirán todos los bloques mediante los identificadores `iso3` y `year`. En esta fase se comprobarán duplicados, periodos, entidades, nombres de variables, tipos de datos y cobertura.

2. **Preparación para modelos sensibles a la escala**

   Esta rama se utilizará para regresión lineal, Ridge, Lasso, Elastic Net, SVR, KNN, redes neuronales y PCA. Podrá incluir imputación, transformación de distribuciones, tratamiento de valores extremos, codificación de variables categóricas y estandarización.

3. **Preparación para modelos basados en árboles**

   Esta rama se utilizará para árboles de regresión, Random Forest, Extra Trees, Gradient Boosting y XGBoost. Mantendrá las variables en su escala original, no aplicará estandarización y realizará un tratamiento más conservador de las distribuciones y de los valores extremos válidos.

Las dos ramas partirán de la misma base maestra, utilizarán la misma variable objetivo y respetarán la misma división temporal entre entrenamiento, validación y prueba.

En este primer apartado se construirá únicamente la base integrada sin imputar, escalar ni modificar todavía las variables explicativas.

In [3]:
# LIBRERÍAS

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display


pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.max_rows",
    200
)

pd.set_option(
    "display.width",
    220
)

pd.set_option(
    "display.float_format",
    lambda value: f"{value:,.4f}"
)

In [4]:
START_YEAR = 2000
PREDICTOR_END_YEAR = 2024
RATING_END_YEAR = 2025

RAW_PATH = Path("data/raw")

PROCESSED_PATH = Path("data/processed")

MODEL_DATA_PATH = Path("data/model")

RAW_PATH.mkdir(
    parents=True,
    exist_ok=True
)

PROCESSED_PATH.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_DATA_PATH.mkdir(
    parents=True,
    exist_ok=True
)


KEY_COLUMNS = [
    "iso3",
    "year"
]


METADATA_COLUMNS = [
    "iso3",
    "iso2",
    "country",
    "region",
    "income_level",
    "year",
    "economic_period"
]


print(
    "Carpeta de trabajo:",
    Path.cwd()
)

print(
    "Carpeta de datos procesados:",
    PROCESSED_PATH
)

print(
    "Carpeta de datos para modelización:",
    MODEL_DATA_PATH
)

print(
    "Periodo de predictores:",
    START_YEAR,
    "-",
    PREDICTOR_END_YEAR
)

print(
    "Periodo de ratings:",
    START_YEAR,
    "-",
    RATING_END_YEAR
)

Carpeta de trabajo: C:\Users\pedro.lopez\Desktop\MASTER\TFM
Carpeta de datos procesados: data\processed
Carpeta de datos para modelización: data\model
Periodo de predictores: 2000 - 2024
Periodo de ratings: 2000 - 2025


In [5]:
# ARCHIVOS DE ENTRADA

DATA_FILES = {

    "macroeconomic": (
        PROCESSED_PATH
        / "wdi_macro_panel_2000_2024.csv"
    ),

    "fiscal": (
        PROCESSED_PATH
        / "imf_fiscal_panel_2000_2024.csv"
    ),

    "external": (
        PROCESSED_PATH
        / "wdi_external_panel_2000_2024.csv"
    ),

    "monetary_financial": (
        PROCESSED_PATH
        / "wdi_monetary_financial_panel_2000_2024.csv"
    ),

    "institutional": (
        PROCESSED_PATH
        / "wgi_institutional_panel_2000_2024.csv"
    ),

    "democracy": (
        PROCESSED_PATH
        / "vdem_democracy_panel_2000_2024.csv"
    ),

    "political_complementary": (
        PROCESSED_PATH
        / "political_complementary_panel_2000_2024.csv"
    ),

    "conflict": (
        PROCESSED_PATH
        / "wdi_conflict_panel_2000_2024.csv"
    ),

    "sovereign_default": (
        PROCESSED_PATH
        / "boc_boe_sovereign_default_panel_2000_2024.csv"
    ),

    "ratings": (
        PROCESSED_PATH
        / "government_credit_rating_mean_2000_2025.csv"
    )
}


file_inventory = pd.DataFrame(
    [
        {
            "block": block_name,
            "file": str(file_path),
            "exists": file_path.exists()
        }
        for block_name, file_path
        in DATA_FILES.items()
    ]
)


display(
    file_inventory
)


missing_files = (
    file_inventory[
        ~file_inventory[
            "exists"
        ]
    ]
)


if not missing_files.empty:

    raise FileNotFoundError(
        "No se encuentran los siguientes archivos:\n\n"
        +
        "\n".join(
            missing_files[
                "file"
            ].tolist()
        )
    )

,block,file,exists
0,macroeconomic,data\processed\wdi_macro_panel_2000_2024.csv,True
1,fiscal,data\processed\imf_fiscal_panel_2000_2024.csv,True
2,external,data\processed\wdi_external_panel_2000_2024.csv,True
3,monetary_financial,data\processed\wdi_monetary_financial_panel_20...,True
4,institutional,data\processed\wgi_institutional_panel_2000_20...,True
5,democracy,data\processed\vdem_democracy_panel_2000_2024.csv,True
6,political_complementary,data\processed\political_complementary_panel_2...,True
7,conflict,data\processed\wdi_conflict_panel_2000_2024.csv,True
8,sovereign_default,data\processed\boc_boe_sovereign_default_panel...,True
9,ratings,data\processed\government_credit_rating_mean_2...,True


In [6]:
# funcion de carga

def load_panel_file(
    block_name,
    file_path
):
    """
    Carga un panel y realiza las comprobaciones
    estructurales mínimas.
    """

    panel = pd.read_csv(
        file_path,
        low_memory=False
    )


    required_columns = {
        "iso3",
        "year"
    }


    missing_required_columns = (
        required_columns
        -
        set(
            panel.columns
        )
    )


    if missing_required_columns:

        raise KeyError(
            f"El bloque '{block_name}' no contiene "
            f"las columnas obligatorias: "
            f"{sorted(missing_required_columns)}"
        )


    panel[
        "iso3"
    ] = (
        panel[
            "iso3"
        ]
        .astype("string")
        .str.strip()
        .str.upper()
    )


    panel[
        "year"
    ] = pd.to_numeric(
        panel[
            "year"
        ],
        errors="coerce"
    ).astype("Int64")


    invalid_keys = panel[
        panel[
            "iso3"
        ].isna()
        |
        panel[
            "year"
        ].isna()
    ]


    if not invalid_keys.empty:

        raise ValueError(
            f"El bloque '{block_name}' contiene "
            f"{len(invalid_keys)} filas sin ISO3 o año válido."
        )


    duplicate_keys = panel.duplicated(
        subset=KEY_COLUMNS,
        keep=False
    )


    if duplicate_keys.any():

        display(
            panel.loc[
                duplicate_keys
            ].sort_values(
                KEY_COLUMNS
            )
        )

        raise ValueError(
            f"El bloque '{block_name}' contiene "
            "duplicados en la clave iso3-año."
        )


    panel = (
        panel
        .sort_values(
            KEY_COLUMNS
        )
        .reset_index(
            drop=True
        )
    )


    return panel

In [7]:
loaded_panels = {}


for block_name, file_path in (
    DATA_FILES.items()
):

    panel = load_panel_file(
        block_name=block_name,
        file_path=file_path
    )

    loaded_panels[
        block_name
    ] = panel

    print(
        f"{block_name}: "
        f"{panel.shape[0]:,} filas, "
        f"{panel.shape[1]:,} columnas, "
        f"{panel['iso3'].nunique():,} entidades, "
        f"{panel['year'].min()}-{panel['year'].max()}"
    )

macroeconomic: 5,425 filas, 16 columnas, 217 entidades, 2000-2024
fiscal: 5,425 filas, 12 columnas, 217 entidades, 2000-2024
external: 5,425 filas, 18 columnas, 217 entidades, 2000-2024
monetary_financial: 5,425 filas, 8 columnas, 217 entidades, 2000-2024
institutional: 5,425 filas, 12 columnas, 217 entidades, 2000-2024
democracy: 5,425 filas, 7 columnas, 217 entidades, 2000-2024
political_complementary: 5,425 filas, 13 columnas, 217 entidades, 2000-2024
conflict: 5,425 filas, 9 columnas, 217 entidades, 2000-2024
sovereign_default: 5,425 filas, 12 columnas, 217 entidades, 2000-2024
ratings: 4,160 filas, 18 columnas, 160 entidades, 2000-2025


In [8]:
panel_audit_rows = []


for block_name, panel in (
    loaded_panels.items()
):

    non_metadata_columns = [
        column
        for column in panel.columns
        if column not in METADATA_COLUMNS
    ]


    panel_audit_rows.append({
        "block": block_name,
        "rows": len(panel),
        "columns": panel.shape[1],
        "entities": panel[
            "iso3"
        ].nunique(),
        "first_year": panel[
            "year"
        ].min(),
        "last_year": panel[
            "year"
        ].max(),
        "duplicate_entity_year": panel.duplicated(
            subset=KEY_COLUMNS
        ).sum(),
        "variables_excluding_metadata": (
            len(non_metadata_columns)
        ),
        "missing_cells_pct": (
            panel[
                non_metadata_columns
            ]
            .isna()
            .mean()
            .mean()
            * 100
            if non_metadata_columns
            else 0
        )
    })


panel_audit = (
    pd.DataFrame(
        panel_audit_rows
    )
    .sort_values(
        "block"
    )
    .reset_index(
        drop=True
    )
)


display(
    panel_audit
)

,block,rows,columns,entities,first_year,last_year,duplicate_entity_year,variables_excluding_metadata,missing_cells_pct
0,conflict,5425,9,217,2000,2024,0,3,85.1060
1,democracy,5425,7,217,2000,2024,0,1,19.5576
2,external,5425,18,217,2000,2024,0,12,31.7481
3,fiscal,5425,12,217,2000,2024,0,6,14.0829
4,institutional,5425,12,217,2000,2024,0,6,10.9923
5,macroeconomic,5425,16,217,2000,2024,0,9,12.7209
6,monetary_financial,5425,8,217,2000,2024,0,2,50.8387
7,political_complementary,5425,13,217,2000,2024,0,7,42.0250
8,ratings,4160,18,160,2000,2025,0,13,15.1461
9,sovereign_default,5425,12,217,2000,2024,0,6,0.0000


In [9]:
macro_panel = (
    loaded_panels[
        "macroeconomic"
    ]
    .copy()
)


master_metadata_columns = [
    column
    for column in METADATA_COLUMNS
    if column in macro_panel.columns
]


master_panel = (
    macro_panel[
        master_metadata_columns
    ]
    .copy()
    .sort_values(
        KEY_COLUMNS
    )
    .reset_index(
        drop=True
    )
)


expected_rows = (
    master_panel[
        "iso3"
    ].nunique()
    *
    (
        PREDICTOR_END_YEAR
        -
        START_YEAR
        +
        1
    )
)


print(
    "Dimensiones de la cuadrícula maestra:",
    master_panel.shape
)

print(
    "Entidades:",
    master_panel[
        "iso3"
    ].nunique()
)

print(
    "Periodo:",
    master_panel[
        "year"
    ].min(),
    "-",
    master_panel[
        "year"
    ].max()
)

print(
    "Filas esperadas:",
    expected_rows
)

print(
    "Duplicados iso3-año:",
    master_panel.duplicated(
        subset=KEY_COLUMNS
    ).sum()
)


if len(master_panel) != expected_rows:

    raise ValueError(
        "La cuadrícula maestra no contiene todas "
        "las combinaciones esperadas entidad-año."
    )

Dimensiones de la cuadrícula maestra: (5425, 7)
Entidades: 217
Periodo: 2000 - 2024
Filas esperadas: 5425
Duplicados iso3-año: 0


In [10]:
def merge_predictor_block(
    master_dataframe,
    block_dataframe,
    block_name
):
    """
    Incorpora las variables propias de un bloque
    mediante una unión uno a uno por iso3 y año.
    """

    repeated_metadata = [
        "iso2",
        "country",
        "region",
        "income_level",
        "economic_period"
    ]


    predictor_columns = [
        column
        for column in block_dataframe.columns
        if column not in (
            KEY_COLUMNS
            +
            repeated_metadata
        )
    ]


    overlapping_columns = (
        set(
            predictor_columns
        )
        &
        set(
            master_dataframe.columns
        )
    )


    if overlapping_columns:

        raise ValueError(
            f"El bloque '{block_name}' contiene "
            f"columnas que ya existen en la base: "
            f"{sorted(overlapping_columns)}"
        )


    block_to_merge = block_dataframe[
        KEY_COLUMNS
        +
        predictor_columns
    ].copy()


    rows_before = len(
        master_dataframe
    )


    merged_dataframe = (
        master_dataframe
        .merge(
            block_to_merge,
            on=KEY_COLUMNS,
            how="left",
            validate="one_to_one"
        )
    )


    rows_after = len(
        merged_dataframe
    )


    if rows_after != rows_before:

        raise RuntimeError(
            f"La unión del bloque '{block_name}' "
            "ha modificado el número de filas."
        )


    print(
        f"{block_name}: "
        f"{len(predictor_columns)} variables añadidas"
    )


    return merged_dataframe


predictor_panel = master_panel.copy()


PREDICTOR_BLOCK_ORDER = [
    "macroeconomic",
    "fiscal",
    "external",
    "monetary_financial",
    "institutional",
    "democracy",
    "political_complementary",
    "conflict",
    "sovereign_default"
]


for block_name in PREDICTOR_BLOCK_ORDER:

    predictor_panel = merge_predictor_block(
        master_dataframe=predictor_panel,
        block_dataframe=loaded_panels[
            block_name
        ],
        block_name=block_name
    )


predictor_panel = (
    predictor_panel
    .sort_values(
        KEY_COLUMNS
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nDimensiones del panel explicativo integrado:",
    predictor_panel.shape
)

print(
    "Duplicados iso3-año:",
    predictor_panel.duplicated(
        subset=KEY_COLUMNS
    ).sum()
)

macroeconomic: 9 variables añadidas
fiscal: 6 variables añadidas
external: 12 variables añadidas
monetary_financial: 2 variables añadidas
institutional: 6 variables añadidas
democracy: 1 variables añadidas
political_complementary: 7 variables añadidas
conflict: 3 variables añadidas
sovereign_default: 6 variables añadidas

Dimensiones del panel explicativo integrado: (5425, 59)
Duplicados iso3-año: 0


In [11]:
# Preparar el objetivo del año siguiente

ratings_panel = loaded_panels["ratings"].copy()


rating_columns = [
    "iso3",
    "year",
    "rating_score_mean",
    "number_of_ratings",
    "has_at_least_one_rating",
    "moodys_score",
    "sp_score",
    "fitch_score",
    "rating_score_min",
    "rating_score_max",
    "rating_score_range",
    "rating_score_std"
]


# Conservar únicamente las columnas que existen

rating_columns = [
    column
    for column in rating_columns
    if column in ratings_panel.columns
]


ratings_target = ratings_panel[
    rating_columns
].copy()


# El año original es el año real del rating

ratings_target = ratings_target.rename(
    columns={
        "year": "rating_year"
    }
)


# El rating de t+1 se relaciona con los predictores de t

ratings_target["predictor_year"] = (
    ratings_target["rating_year"] - 1
)


# Añadir el sufijo t1 a las variables del objetivo

columns_to_rename = [
    column
    for column in ratings_target.columns
    if column not in [
        "iso3",
        "rating_year",
        "predictor_year"
    ]
]


ratings_target = ratings_target.rename(
    columns={
        column: f"{column}_t1"
        for column in columns_to_rename
    }
)


# Conservar los años compatibles con los predictores

ratings_target = ratings_target[
    ratings_target["predictor_year"].between(
        START_YEAR,
        PREDICTOR_END_YEAR
    )
].copy()


ratings_target = ratings_target.sort_values(
    [
        "iso3",
        "predictor_year"
    ]
).reset_index(drop=True)


# Comprobar la relación temporal

incorrect_alignment = (
    ratings_target["rating_year"]
    !=
    ratings_target["predictor_year"] + 1
)


print(
    "Relaciones temporales incorrectas:",
    incorrect_alignment.sum()
)

print(
    "Periodo de los predictores:",
    ratings_target["predictor_year"].min(),
    "-",
    ratings_target["predictor_year"].max()
)

print(
    "Periodo de los ratings objetivo:",
    ratings_target["rating_year"].min(),
    "-",
    ratings_target["rating_year"].max()
)

print(
    "Duplicados iso3-año:",
    ratings_target.duplicated(
        subset=[
            "iso3",
            "predictor_year"
        ]
    ).sum()
)


if incorrect_alignment.any():

    raise ValueError(
        "Hay ratings que no están exactamente "
        "un año después de los predictores."
    )


if ratings_target.duplicated(
    subset=[
        "iso3",
        "predictor_year"
    ]
).any():

    raise ValueError(
        "Hay varios ratings para una misma "
        "entidad y año predictor."
    )


# Unir los predictores de t con el rating de t+1

integrated_panel_raw = predictor_panel.merge(
    ratings_target,
    left_on=[
        "iso3",
        "year"
    ],
    right_on=[
        "iso3",
        "predictor_year"
    ],
    how="left",
    validate="one_to_one"
)


integrated_panel_raw = integrated_panel_raw.sort_values(
    [
        "iso3",
        "year"
    ]
).reset_index(drop=True)


# Comprobar la relación después de la unión

rows_with_target = (
    integrated_panel_raw["rating_year"].notna()
)


incorrect_merge_alignment = (
    integrated_panel_raw.loc[
        rows_with_target,
        "rating_year"
    ]
    !=
    integrated_panel_raw.loc[
        rows_with_target,
        "year"
    ]
    + 1
)


print(
    "Relaciones incorrectas después de la unión:",
    incorrect_merge_alignment.sum()
)


if incorrect_merge_alignment.any():

    raise ValueError(
        "La unión entre predictores y rating "
        "no respeta la relación t con t+1."
    )


# predictor_year es igual a year y ya no es necesario

integrated_panel_raw = integrated_panel_raw.drop(
    columns=[
        "predictor_year"
    ]
)


# Indicar si el objetivo está disponible

integrated_panel_raw["target_available_t1"] = (
    integrated_panel_raw[
        "rating_score_mean_t1"
    ]
    .notna()
    .astype("Int64")
)


print(
    "Dimensiones de la base integrada:",
    integrated_panel_raw.shape
)

print(
    "Entidades:",
    integrated_panel_raw["iso3"].nunique()
)

print(
    "Entidades-año totales:",
    len(integrated_panel_raw)
)

print(
    "Entidades-año con objetivo:",
    integrated_panel_raw[
        "target_available_t1"
    ].sum()
)

print(
    "Porcentaje con objetivo:",
    round(
        integrated_panel_raw[
            "target_available_t1"
        ].mean()
        * 100,
        2
    ),
    "%"
)

print(
    "Duplicados iso3-año:",
    integrated_panel_raw.duplicated(
        subset=[
            "iso3",
            "year"
        ]
    ).sum()
)


# Guardar la base integrada sin limpiar

INTEGRATED_RAW_FILE = (
    MODEL_DATA_PATH
    / "country_risk_panel_integrated_raw_2000_2024.csv"
)


integrated_panel_raw.to_csv(
    INTEGRATED_RAW_FILE,
    index=False,
    encoding="utf-8-sig"
)


print(
    "Base integrada guardada en:",
    INTEGRATED_RAW_FILE
)

Relaciones temporales incorrectas: 0
Periodo de los predictores: 2000 - 2024
Periodo de los ratings objetivo: 2001 - 2025
Duplicados iso3-año: 0
Relaciones incorrectas después de la unión: 0
Dimensiones de la base integrada: (5425, 71)
Entidades: 217
Entidades-año totales: 5425
Entidades-año con objetivo: 3434
Porcentaje con objetivo: 63.3 %
Duplicados iso3-año: 0
Base integrada guardada en: data\model\country_risk_panel_integrated_raw_2000_2024.csv


---
## 2. Auditoría inicial de la base integrada

Antes de aplicar transformaciones o eliminar observaciones, se analiza la estructura de la base integrada.

Para cada variable se estudian:

- El tipo de dato.
- El número de valores diferentes.
- El número y porcentaje de valores ausentes.
- La cobertura dentro de la muestra que dispone de variable objetivo.
- La posible existencia de variables constantes.
- La presencia de valores infinitos.

Esta auditoría permitirá decidir posteriormente qué variables deben conservarse, transformarse, imputarse o eliminarse.

En esta fase no se modifica todavía el dataset.


In [12]:
# Definir los grupos de columnas

metadata_columns = [
    "iso3",
    "iso2",
    "country",
    "region",
    "income_level",
    "year",
    "economic_period"
]

metadata_columns = [
    column
    for column in metadata_columns
    if column in integrated_panel_raw.columns
]


target_columns = [
    column
    for column in integrated_panel_raw.columns
    if column.endswith("_t1")
]

target_columns += [
    column
    for column in [
        "rating_year",
        "target_available_t1"
    ]
    if column in integrated_panel_raw.columns
]

target_columns = list(
    dict.fromkeys(target_columns)
)


predictor_columns = [
    column
    for column in integrated_panel_raw.columns
    if column not in metadata_columns + target_columns
]


print(
    "Variables identificativas:",
    len(metadata_columns)
)

print(
    "Variables explicativas:",
    len(predictor_columns)
)

print(
    "Variables relacionadas con el objetivo:",
    len(target_columns)
)

Variables identificativas: 7
Variables explicativas: 52
Variables relacionadas con el objetivo: 12


In [13]:
# Crear la muestra con objetivo disponible

model_sample = integrated_panel_raw[
    integrated_panel_raw[
        "target_available_t1"
    ] == 1
].copy()


print(
    "Filas de la base completa:",
    len(integrated_panel_raw)
)

print(
    "Filas de la muestra con objetivo:",
    len(model_sample)
)

Filas de la base completa: 5425
Filas de la muestra con objetivo: 3434


In [14]:
# Auditar cada variable

variable_audit = pd.DataFrame({
    "variable": predictor_columns,

    "dtype": [
        str(
            integrated_panel_raw[column].dtype
        )
        for column in predictor_columns
    ],

    "unique_values": [
        integrated_panel_raw[
            column
        ].nunique(
            dropna=True
        )
        for column in predictor_columns
    ],

    "missing_full": [
        integrated_panel_raw[
            column
        ].isna().sum()
        for column in predictor_columns
    ],

    "missing_full_pct": [
        integrated_panel_raw[
            column
        ].isna().mean() * 100
        for column in predictor_columns
    ],

    "missing_model_sample": [
        model_sample[
            column
        ].isna().sum()
        for column in predictor_columns
    ],

    "missing_model_sample_pct": [
        model_sample[
            column
        ].isna().mean() * 100
        for column in predictor_columns
    ]
})

In [15]:
# Detectar variables constantes

variable_audit[
    "constant_variable"
] = (
    variable_audit[
        "unique_values"
    ] <= 1
)

In [16]:
# Detectar valores infinitos en variables numéricas

numeric_predictors = (
    integrated_panel_raw[
        predictor_columns
    ]
    .select_dtypes(
        include="number"
    )
    .columns
    .tolist()
)


infinite_counts = {}


for column in numeric_predictors:

    values = pd.to_numeric(
        integrated_panel_raw[column],
        errors="coerce"
    )

    infinite_counts[column] = (
        np.isinf(values).sum()
    )


variable_audit[
    "infinite_values"
] = (
    variable_audit[
        "variable"
    ]
    .map(
        infinite_counts
    )
    .fillna(0)
    .astype(int)
)

In [17]:
# Detectar valores infinitos en variables numéricas

numeric_predictors = (
    integrated_panel_raw[
        predictor_columns
    ]
    .select_dtypes(
        include="number"
    )
    .columns
    .tolist()
)


infinite_counts = {}


for column in numeric_predictors:

    values = pd.to_numeric(
        integrated_panel_raw[column],
        errors="coerce"
    )

    infinite_counts[column] = (
        np.isinf(values).sum()
    )


variable_audit[
    "infinite_values"
] = (
    variable_audit[
        "variable"
    ]
    .map(
        infinite_counts
    )
    .fillna(0)
    .astype(int)
)

In [18]:
# Resumen de problemas detectados

print(
    "Variables constantes:",
    variable_audit[
        "constant_variable"
    ].sum()
)

print(
    "Variables con valores infinitos:",
    (
        variable_audit[
            "infinite_values"
        ] > 0
    ).sum()
)

print(
    "Variables con más del 40 % de ausentes "
    "en la muestra del modelo:",
    (
        variable_audit[
            "missing_model_sample_pct"
        ] > 40
    ).sum()
)

print(
    "Variables completamente vacías "
    "en la muestra del modelo:",
    (
        variable_audit[
            "missing_model_sample_pct"
        ] == 100
    ).sum()
)

Variables constantes: 0
Variables con valores infinitos: 0
Variables con más del 40 % de ausentes en la muestra del modelo: 10
Variables completamente vacías en la muestra del modelo: 0


In [19]:
# Guardar la auditoría

VARIABLE_AUDIT_FILE = (
    MODEL_DATA_PATH
    / "country_risk_variable_audit.csv"
)


variable_audit.to_csv(
    VARIABLE_AUDIT_FILE,
    index=False,
    encoding="utf-8-sig"
)


print(
    "Auditoría guardada en:",
    VARIABLE_AUDIT_FILE
)

Auditoría guardada en: data\model\country_risk_variable_audit.csv


---
## 3. Análisis de valores ausentes

Se analiza la cobertura de las variables explicativas dentro de la muestra que dispone de calificación crediticia en el año siguiente.

Las variables se clasifican inicialmente según su porcentaje de valores ausentes:

- Sin valores ausentes.
- Hasta el 10 %.
- Entre el 10 % y el 20 %.
- Entre el 20 % y el 40 %.
- Más del 40 %.
- Completamente vacías.

Esta clasificación es únicamente descriptiva. Las variables no se eliminan automáticamente, ya que los valores ausentes pueden deberse a una menor cobertura temporal, a diferencias entre fuentes o a características específicas de determinados países.

También se estudia el primer y último año con información disponible para cada variable.

In [20]:
# Clasificar las variables por porcentaje de ausentes

def classify_missing_percentage(value):

    if value == 0:
        return "0 %"

    if value <= 10:
        return "0-10 %"

    if value <= 20:
        return "10-20 %"

    if value <= 40:
        return "20-40 %"

    if value < 100:
        return "Más del 40 %"

    return "100 %"


variable_audit["missing_group"] = (
    variable_audit["missing_model_sample_pct"]
    .apply(classify_missing_percentage)
)

In [21]:
# Resumen por grupos de valores ausentes

missing_group_order = [
    "0 %",
    "0-10 %",
    "10-20 %",
    "20-40 %",
    "Más del 40 %",
    "100 %"
]


missing_summary = (
    variable_audit["missing_group"]
    .value_counts()
    .reindex(
        missing_group_order,
        fill_value=0
    )
    .rename_axis("missing_group")
    .reset_index(name="number_of_variables")
)


display(missing_summary)

,missing_group,number_of_variables
0,0 %,6
1,0-10 %,28
2,10-20 %,3
3,20-40 %,5
4,Más del 40 %,10
5,100 %,0


In [22]:
# Variables con más del 20 % de valores ausentes

high_missing_variables = variable_audit[
    variable_audit["missing_model_sample_pct"] > 20
][
    [
        "variable",
        "dtype",
        "unique_values",
        "missing_full_pct",
        "missing_model_sample_pct",
        "constant_variable",
        "infinite_values"
    ]
].copy()


display(high_missing_variables)

,variable,dtype,unique_values,missing_full_pct,missing_model_sample_pct,constant_variable,infinite_values
18,external_debt_pct_gni,float64,2916,46.2488,47.5830,False,0
19,external_debt_service_pct_exports,float64,2748,49.3456,48.6313,False,0
20,external_debt_stocks_current_usd,float64,2958,45.4747,47.4083,False,0
24,reserves_pct_external_debt,float64,2613,51.8341,52.6209,False,0
25,short_term_debt_pct_reserves,float64,2551,51.8341,52.6209,False,0
27,bank_npl_pct_gross_loans,float64,2260,58.2120,44.7292,False,0
28,real_interest_rate_pct,float64,3067,43.4654,37.0996,False,0
36,head_of_government,str,826,47.6129,39.6040,False,0
37,head_of_government_tenure_years,float64,23,47.6129,39.6040,False,0
38,average_government_duration_since_2000,float64,154,45.8802,38.4391,False,0


In [23]:
# Calcular la cobertura temporal de cada variable

coverage_rows = []


for column in predictor_columns:

    available_data = model_sample[
        model_sample[column].notna()
    ]

    if available_data.empty:

        first_year = pd.NA
        last_year = pd.NA
        available_rows = 0

    else:

        first_year = available_data["year"].min()
        last_year = available_data["year"].max()
        available_rows = len(available_data)


    coverage_rows.append({
        "variable": column,
        "first_year_available": first_year,
        "last_year_available": last_year,
        "available_rows": available_rows,
        "available_pct": (
            available_rows
            / len(model_sample)
            * 100
        )
    })


temporal_coverage = pd.DataFrame(
    coverage_rows
)

In [24]:
# Añadir la cobertura temporal a la auditoría

variable_audit = variable_audit.merge(
    temporal_coverage,
    on="variable",
    how="left",
    validate="one_to_one"
)


variable_audit = variable_audit.sort_values(
    "missing_model_sample_pct",
    ascending=False
).reset_index(drop=True)


display(
    variable_audit[
        [
            "variable",
            "dtype",
            "unique_values",
            "missing_model_sample_pct",
            "first_year_available",
            "last_year_available",
            "available_pct"
        ]
    ]
)

,variable,dtype,unique_values,missing_model_sample_pct,first_year_available,last_year_available,available_pct
0,log_battle_related_deaths,float64,540,85.6727,2000,2024,14.3273
1,battle_related_deaths,float64,540,85.6727,2000,2024,14.3273
2,armed_conflict_dummy,float64,2,85.6727,2000,2024,14.3273
3,global_competitiveness_index,float64,290,62.0850,2007,2017,37.9150
4,reserves_pct_external_debt,float64,2613,52.6209,2000,2024,47.3791
5,short_term_debt_pct_reserves,float64,2551,52.6209,2000,2024,47.3791
6,external_debt_service_pct_exports,float64,2748,48.6313,2000,2024,51.3687
7,external_debt_pct_gni,float64,2916,47.5830,2000,2024,52.4170
8,external_debt_stocks_current_usd,float64,2958,47.4083,2000,2024,52.5917
9,bank_npl_pct_gross_loans,float64,2260,44.7292,2001,2024,55.2708


In [25]:
# Porcentaje de ausentes por año y variable

missing_by_year = (
    model_sample
    .groupby("year")[predictor_columns]
    .agg(
        lambda values:
        values.isna().mean() * 100
    )
)


display(
    missing_by_year.round(2)
)

,exchange_rate_lcu_per_usd,gdp_growth_pct,gdp_per_capita_constant_2015_usd,gdp_per_capita_ppp_constant_2021_intl_usd,gdp_ppp_constant_2021_intl_usd,gross_savings_pct_gdp,inflation_cpi_pct,investment_pct_gdp,unemployment_pct,fiscal_balance_pct_gdp,government_debt_pct_gdp,government_expenditure_pct_gdp,government_revenue_pct_gdp,primary_balance_pct_gdp,net_interest_payments_pct_gdp,current_account_balance_pct_gdp,exports_growth_pct,exports_pct_gdp,external_debt_pct_gni,external_debt_service_pct_exports,external_debt_stocks_current_usd,fdi_net_inflows_pct_gdp,imports_pct_gdp,reserves_months_imports,reserves_pct_external_debt,short_term_debt_pct_reserves,trade_openness_pct_gdp,bank_npl_pct_gross_loans,real_interest_rate_pct,control_of_corruption_estimate,government_effectiveness_estimate,political_stability_estimate,regulatory_quality_estimate,rule_of_law_estimate,voice_accountability_estimate,electoral_democracy_index,head_of_government,head_of_government_tenure_years,average_government_duration_since_2000,government_change_dummy,government_changes_previous_5y,global_competitiveness_index,fragile_states_index,battle_related_deaths,armed_conflict_dummy,log_battle_related_deaths,sovereign_debt_in_default_usd_mn,sovereign_default_dummy,default_previous_5y,default_previous_10y,years_since_last_default,never_default_history_dummy
year,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2000,1.0000,1.0000,2.0000,5.0000,5.0000,16.0000,11.0000,7.0000,4.0000,9.0000,11.0000,9.0000,8.0000,12.0000,12.0000,13.0000,15.0000,8.0000,60.0000,62.0000,60.0000,5.0000,8.0000,14.0000,62.0000,62.0000,8.0000,100.0000,39.0000,2.0000,2.0000,2.0000,2.0000,1.0000,1.0000,7.0000,38.0000,38.0000,38.0000,7.0000,7.0000,100.0000,100.0000,89.0000,89.0000,89.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2001,0.9700,0.9700,1.9400,4.8500,4.8500,18.4500,9.7100,6.8000,3.8800,5.8300,5.8300,5.8300,4.8500,8.7400,8.7400,14.5600,12.6200,7.7700,58.2500,61.1700,58.2500,4.8500,7.7700,15.5300,61.1700,61.1700,7.7700,99.0300,37.8600,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,6.8000,39.8100,39.8100,37.8600,6.8000,6.8000,100.0000,100.0000,88.3500,88.3500,88.3500,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2002,1.8500,0.9300,1.8500,4.6300,4.6300,17.5900,10.1900,8.3300,3.7000,4.6300,4.6300,4.6300,4.6300,7.4100,7.4100,12.0400,13.8900,8.3300,55.5600,57.4100,55.5600,3.7000,8.3300,12.9600,58.3300,58.3300,8.3300,99.0700,39.8100,1.8500,1.8500,1.8500,1.8500,0.9300,0.9300,6.4800,38.8900,38.8900,38.8900,6.4800,6.4800,100.0000,100.0000,91.6700,91.6700,91.6700,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2003,1.7700,0.8800,1.7700,4.4200,4.4200,17.7000,10.6200,8.8500,3.5400,4.4200,4.4200,4.4200,4.4200,7.0800,7.0800,12.3900,14.1600,8.8500,53.9800,54.8700,53.9800,3.5400,8.8500,15.0400,59.2900,59.2900,8.8500,97.3500,40.7100,1.7700,1.7700,0.8800,1.7700,0.8800,0.8800,6.1900,38.9400,38.9400,38.9400,6.1900,6.1900,100.0000,100.0000,91.1500,91.1500,91.1500,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2004,1.6800,0.8400,1.6800,4.2000,4.2000,17.6500,9.2400,8.4000,3.3600,3.3600,4.2000,3.3600,3.3600,5.8800,5.8800,12.6100,14.2900,8.4000,52.1000,54.6200,52.1000,4.2000,8.4000,15.1300,57.9800,57.9800,8.4000,97.4800,36.1300,1.6800,1.6800,0.8400,1.6800,0.8400,0.8400,5.8800,39.5000,39.5000,39.5000,5.8800,5.8800,100.0000,100.0000,89.9200,89.9200,89.9200,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2005,1.6400,0.8200,1.6400,4.1000,4.1000,16.3900,8.2000,9.8400,3.2800,3.2800,4.1000,3.2800,3.2800,4.9200,4.9200,9.0200,14.7500,9.8400,51.6400,53.2800,50.8200,4.1000,9.8400,12.3000,56.5600,56.5600,9.8400,79.5100,31.1500,1.6400,1.6400,0.8200,1.6400,0.8200,0.8200,5.7400,39.3400,39.3400,39.3400,5.7400,5.7400,100.0000,100.0000,86.8900,86.8900,86.8900,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2006,1.5500,0.7800,0.7800,3.1000,3.1000,16.2800,6.9800,9.3000,3.1000,3.1000,3.1000,3.1000,3.1000,4.6500,4.6500,9.3000,15.5000,9.3000,48.8400,51.1600,48.0600,3.1000,9.3000,12.4000,52.7100,52.7100,9.3000,77.5200,30.2300,1.5500,1.5500,0.7800,1.

In [26]:
# Resumen de valores ausentes por año

annual_missing_summary = pd.DataFrame({
    "year": missing_by_year.index,

    "average_missing_pct": (
        missing_by_year.mean(axis=1)
    ),

    "variables_above_20_pct": (
        missing_by_year.gt(20).sum(axis=1)
    ),

    "variables_above_40_pct": (
        missing_by_year.gt(40).sum(axis=1)
    ),

    "completely_missing_variables": (
        missing_by_year.eq(100).sum(axis=1)
    )
}).reset_index(drop=True)


display(
    annual_missing_summary.round(2)
)

,year,average_missing_pct,variables_above_20_pct,variables_above_40_pct,completely_missing_variables
0,2000,23.8700,15,11,3
1,2001,34.6700,21,17,8
2,2002,23.0600,15,11,2
3,2003,22.9600,15,12,2
4,2004,22.5100,15,11,2
5,2005,21.6600,15,11,2
6,2006,19.7100,14,10,1
7,2007,17.4000,13,9,0
8,2008,16.9400,13,9,0
9,2009,17.3200,13,9,0


In [27]:
# Variables constantes

constant_variables = variable_audit.loc[
    variable_audit["constant_variable"],
    [
        "variable",
        "unique_values",
        "missing_model_sample_pct"
    ]
]


display(constant_variables)

,variable,unique_values,missing_model_sample_pct


In [28]:
# Variables con valores infinitos

variables_with_infinite_values = variable_audit.loc[
    variable_audit["infinite_values"] > 0,
    [
        "variable",
        "infinite_values",
        "missing_model_sample_pct"
    ]
]


display(variables_with_infinite_values)

,variable,infinite_values,missing_model_sample_pct


In [29]:
# Guardar la auditoría actualizada

variable_audit.to_csv(
    VARIABLE_AUDIT_FILE,
    index=False,
    encoding="utf-8-sig"
)


MISSING_BY_YEAR_FILE = (
    MODEL_DATA_PATH
    / "country_risk_missing_values_by_year.csv"
)


missing_by_year.to_csv(
    MISSING_BY_YEAR_FILE,
    encoding="utf-8-sig"
)


print(
    "Auditoría actualizada guardada en:",
    VARIABLE_AUDIT_FILE
)

print(
    "Cobertura anual guardada en:",
    MISSING_BY_YEAR_FILE
)

Auditoría actualizada guardada en: data\model\country_risk_variable_audit.csv
Cobertura anual guardada en: data\model\country_risk_missing_values_by_year.csv


---
## 4. Incorporación del rating actual

Además del rating objetivo del año siguiente, se incorpora el rating observado en el año de las variables explicativas.

Esto permite construir tres planteamientos:

- Modelo basado únicamente en variables económicas, fiscales, externas, institucionales y políticas.
- Modelo que añade el rating actual como predictor.
- Modelo de persistencia que utiliza el rating actual como predicción del rating del año siguiente.

También se calcula el cambio anual del rating como la diferencia entre el rating futuro y el rating actual.

In [38]:
# Preparar el rating actual

current_rating_columns = [
    "iso3",
    "year",
    "rating_score_mean",
    "number_of_ratings",
    "has_at_least_one_rating"
]

current_rating_columns = [
    column
    for column in current_rating_columns
    if column in ratings_panel.columns
]


if "rating_score_mean" not in current_rating_columns:
    raise KeyError(
        "No se encuentra la variable rating_score_mean."
    )


ratings_current = ratings_panel[
    current_rating_columns
].copy()


ratings_current = ratings_current.rename(
    columns={
        column: f"{column}_t"
        for column in ratings_current.columns
        if column not in ["iso3", "year"]
    }
)

display(ratings_current.head())

,iso3,year,rating_score_mean_t,number_of_ratings_t,has_at_least_one_rating_t
0,ABW,2000,NaN,0,0
1,ABW,2001,NaN,0,0
2,ABW,2002,NaN,0,0
3,ABW,2003,NaN,0,0
4,ABW,2004,NaN,0,0


In [36]:
print(
    "Ratings disponibles:",
    ratings_current["rating_score_mean_t"].notna().sum()
)

Ratings disponibles: 3558


In [39]:
# Añadir el rating actual a la base integrada

clean_panel = integrated_panel_raw.merge(
    ratings_current,
    on=[
        "iso3",
        "year"
    ],
    how="left",
    validate="one_to_one"
)


clean_panel = clean_panel.sort_values(
    [
        "iso3",
        "year"
    ]
).reset_index(drop=True)

In [40]:
# Indicar si existe rating actual

clean_panel["rating_available_t"] = (
    clean_panel["rating_score_mean_t"]
    .notna()
    .astype("Int64")
)


# Calcular el cambio entre t y t+1

clean_panel["rating_change_t1"] = (
    clean_panel["rating_score_mean_t1"]
    -
    clean_panel["rating_score_mean_t"]
)


# Indicar si puede calcularse el baseline de persistencia

clean_panel["persistence_available"] = (
    clean_panel[
        [
            "rating_score_mean_t",
            "rating_score_mean_t1"
        ]
    ]
    .notna()
    .all(axis=1)
    .astype("Int64")
)

In [41]:
# Comprobar el resultado

print(
    "Dimensiones:",
    clean_panel.shape
)

print(
    "Filas con rating actual:",
    clean_panel["rating_available_t"].sum()
)

print(
    "Filas con rating futuro:",
    clean_panel["target_available_t1"].sum()
)

print(
    "Filas con rating actual y futuro:",
    clean_panel["persistence_available"].sum()
)

print(
    "Cambios de rating calculados:",
    clean_panel["rating_change_t1"].notna().sum()
)

print(
    "Duplicados iso3-año:",
    clean_panel.duplicated(
        subset=[
            "iso3",
            "year"
        ]
    ).sum()
)

Dimensiones: (5425, 77)
Filas con rating actual: 3374
Filas con rating futuro: 3434
Filas con rating actual y futuro: 3373
Cambios de rating calculados: 3373
Duplicados iso3-año: 0


In [43]:
display(
    clean_panel[
        [
            "iso3",
            "year",
            "rating_score_mean_t",
            "rating_year",
            "rating_score_mean_t1",
            "rating_change_t1"
        ]
    ]
    .dropna(
        subset=[
            "rating_score_mean_t",
            "rating_score_mean_t1"
        ]
    )
    .head(10)
)

,iso3,year,rating_score_mean_t,rating_year,rating_score_mean_t1,rating_change_t1
10,ABW,2010,70.0000,2011,70.0000,0.0000
11,ABW,2011,70.0000,2012,70.0000,0.0000
12,ABW,2012,70.0000,2013,65.0000,-5.0000
13,ABW,2013,65.0000,2014,65.0000,0.0000
14,ABW,2014,65.0000,2015,65.0000,0.0000
15,ABW,2015,65.0000,2016,65.0000,0.0000
16,ABW,2016,65.0000,2017,65.0000,0.0000
17,ABW,2017,65.0000,2018,65.0000,0.0000
18,ABW,2018,65.0000,2019,65.0000,0.0000
19,ABW,2019,65.0000,2020,65.0000,0.0000


Lo que se ha hecho es:
    
        rating_score_mean_t     → rating observado en el año de los predictores
        rating_score_mean_t1    → rating que queremos predecir
        rating_change_t1        → cambio entre ambos años

Por ejemplo:
$$
\text{RatingChange}_{2025}
=
\text{Rating}_{2025}
-
\text{Rating}_{2024}
$$

## 5. Selección estructural de variables

Antes de aplicar métodos de imputación o transformación, se clasifican las variables según su utilidad y cobertura.

Se definen tres grupos:

- Variables principales: utilizadas en la comparación común de los modelos.
- Variables ampliadas: conservadas para análisis complementarios debido a su menor cobertura.
- Variables excluidas: identificadores, variables con demasiadas categorías o variables con cobertura temporal insuficiente.

Los valores ausentes todavía no se imputan.

In [45]:
# Variables que no se utilizarán como predictores

excluded_variables = [
    "head_of_government",
    "global_competitiveness_index"
]

excluded_variables = [
    column
    for column in excluded_variables
    if column in clean_panel.columns
]


# Variables reservadas para el modelo ampliado

extended_variables = [
    "head_of_government_tenure_years",
    "average_government_duration_since_2000",
    "fragile_states_index",
    "real_interest_rate_pct",
    "external_debt_pct_gni",
    "external_debt_service_pct_exports",
    "external_debt_stocks_current_usd",
    "reserves_pct_external_debt",
    "short_term_debt_pct_reserves",
    "bank_npl_pct_gross_loans",
    "battle_related_deaths",
    "armed_conflict_dummy",
    "log_battle_related_deaths"
]

extended_variables = [
    column
    for column in extended_variables
    if column in clean_panel.columns
]

In [46]:
# Variables que no deben considerarse explicativas

non_predictor_columns = list(
    dict.fromkeys(
        metadata_columns
        + target_columns
        + [
            "rating_score_mean_t",
            "number_of_ratings_t",
            "has_at_least_one_rating_t",
            "rating_available_t",
            "rating_change_t1",
            "persistence_available"
        ]
    )
)

non_predictor_columns = [
    column
    for column in non_predictor_columns
    if column in clean_panel.columns
]

In [47]:
# Obtener todas las variables explicativas disponibles

all_predictor_columns = [
    column
    for column in clean_panel.columns
    if column not in non_predictor_columns
]

In [48]:
# Variables del modelo principal

core_variables = [
    column
    for column in all_predictor_columns
    if column not in excluded_variables
    and column not in extended_variables
]

In [49]:
# Comprobar los grupos

print(
    "Variables principales:",
    len(core_variables)
)

print(
    "Variables ampliadas:",
    len(extended_variables)
)

print(
    "Variables excluidas:",
    len(excluded_variables)
)


print(
    "\nVariables ampliadas:"
)

print(
    extended_variables
)


print(
    "\nVariables excluidas:"
)

print(
    excluded_variables
)

Variables principales: 37
Variables ampliadas: 13
Variables excluidas: 2

Variables ampliadas:
['head_of_government_tenure_years', 'average_government_duration_since_2000', 'fragile_states_index', 'real_interest_rate_pct', 'external_debt_pct_gni', 'external_debt_service_pct_exports', 'external_debt_stocks_current_usd', 'reserves_pct_external_debt', 'short_term_debt_pct_reserves', 'bank_npl_pct_gross_loans', 'battle_related_deaths', 'armed_conflict_dummy', 'log_battle_related_deaths']

Variables excluidas:
['head_of_government', 'global_competitiveness_index']


In [50]:
# Comprobar que los grupos no se solapan

core_extended_overlap = set(
    core_variables
).intersection(
    extended_variables
)

core_excluded_overlap = set(
    core_variables
).intersection(
    excluded_variables
)

extended_excluded_overlap = set(
    extended_variables
).intersection(
    excluded_variables
)


print(
    "Coincidencias principal-ampliado:",
    len(core_extended_overlap)
)

print(
    "Coincidencias principal-excluido:",
    len(core_excluded_overlap)
)

print(
    "Coincidencias ampliado-excluido:",
    len(extended_excluded_overlap)
)

Coincidencias principal-ampliado: 0
Coincidencias principal-excluido: 0
Coincidencias ampliado-excluido: 0


In [51]:
# El modelo ampliado incluye las variables principales y ampliadas

all_model_variables = (
    core_variables
    +
    extended_variables
)


print(
    "Variables del modelo principal:",
    len(core_variables)
)

print(
    "Variables del modelo ampliado:",
    len(all_model_variables)
)

Variables del modelo principal: 37
Variables del modelo ampliado: 50


In [52]:
# Crear una tabla con la clasificación de cada variable

variable_groups = pd.DataFrame({
    "variable": (
        core_variables
        +
        extended_variables
        +
        excluded_variables
    ),

    "group": (
        ["core"] * len(core_variables)
        +
        ["extended"] * len(extended_variables)
        +
        ["excluded"] * len(excluded_variables)
    )
})


variable_groups = variable_groups.sort_values(
    [
        "group",
        "variable"
    ]
).reset_index(drop=True)


display(variable_groups)

,variable,group
0,control_of_corruption_estimate,core
1,current_account_balance_pct_gdp,core
2,default_previous_10y,core
3,default_previous_5y,core
4,electoral_democracy_index,core
5,exchange_rate_lcu_per_usd,core
6,exports_growth_pct,core
7,exports_pct_gdp,core
8,fdi_net_inflows_pct_gdp,core
9,fiscal_balance_pct_gdp,core


In [53]:
VARIABLE_GROUPS_FILE = (
    MODEL_DATA_PATH
    / "country_risk_variable_groups.csv"
)


variable_groups.to_csv(
    VARIABLE_GROUPS_FILE,
    index=False,
    encoding="utf-8-sig"
)


print(
    "Clasificación guardada en:",
    VARIABLE_GROUPS_FILE
)

Clasificación guardada en: data\model\country_risk_variable_groups.csv


Con esta clasificación no se está eliminando físicamente columnas de clean_panel. Solamente se deja decidido qué variables utilizará cada versión:

    Modelo principal=variables con cobertura suficiente
    Modelo ampliado=variables principales+variables con menor cobertura

## 6. Revisión de rangos y valores imposibles

Se revisan los valores mínimos y máximos de las variables numéricas para detectar posibles errores.

No se consideran errores los valores extremos que sean económicamente posibles. Por ejemplo, una deuda pública superior al 100 % del PIB puede ser elevada, pero no necesariamente incorrecta.

En esta fase solo se detectan valores incompatibles con la definición de cada variable.

In [54]:
# Seleccionar las variables numéricas

numeric_variables = (
    clean_panel[all_model_variables]
    .select_dtypes(include="number")
    .columns
    .tolist()
)


# Revisar los principales valores de cada variable

range_summary = (
    clean_panel[numeric_variables]
    .describe(
        percentiles=[
            0.01,
            0.50,
            0.99
        ]
    )
    .T
)


range_summary = range_summary[
    [
        "count",
        "min",
        "1%",
        "50%",
        "99%",
        "max"
    ]
].round(3)


display(range_summary)

,count,min,1%,50%,99%,max
exchange_rate_lcu_per_usd,"5,170.0000",0.0440,0.3760,6.9230,"14,393.1020","6,723,052,073.3380"
gdp_growth_pct,"5,168.0000",-54.4020,-14.8210,3.5220,17.6120,86.8270
gdp_per_capita_constant_2015_usd,"5,159.0000",233.0320,330.6240,"5,724.4320","106,811.5520","247,170.1040"
gdp_per_capita_ppp_constant_2021_intl_usd,"4,916.0000",702.8500,"1,028.4430","14,329.1810","118,540.6380","174,569.5230"
gdp_ppp_constant_2021_intl_usd,"4,916.0000","36,607,944.0870","150,126,431.1130","64,279,616,707.4850","10,019,665,558,416.7656","33,592,044,935,159.1016"
gross_savings_pct_gdp,"3,827.0000",-39.1730,-6.3790,21.4870,59.5920,372.9750
inflation_cpi_pct,"4,519.0000",-16.8600,-2.1860,3.4940,53.4910,557.2020
investment_pct_gdp,"4,273.0000",-15.6780,3.0510,23.1150,49.6830,76.7820
unemployment_pct,"4,666.0000",0.1000,0.5800,6.1280,27.5980,37.3200
fiscal_balance_pct_gdp,"4,714.0000",-55.7490,-17.7730,-2.3060,23.8750,125.1350


In [55]:
# Buscar variables binarias por su nombre

binary_variables = [
    column
    for column in all_model_variables
    if column.endswith(
        (
            "_dummy",
            "_flag"
        )
    )
]


binary_audit = pd.DataFrame({
    "variable": binary_variables,

    "values": [
        sorted(
            clean_panel[column]
            .dropna()
            .unique()
            .tolist()
        )
        for column in binary_variables
    ],

    "invalid_values": [
        (
            clean_panel[column]
            .notna()
            &
            ~clean_panel[column].isin(
                [
                    0,
                    1
                ]
            )
        ).sum()
        for column in binary_variables
    ]
})


display(binary_audit)

,variable,values,invalid_values
0,government_change_dummy,"[0.0, 1.0]",0
1,sovereign_default_dummy,"[0, 1]",0
2,never_default_history_dummy,"[0, 1]",0
3,armed_conflict_dummy,"[0.0, 1.0]",0


In [56]:
# Definir los rangos que son necesariamente válidos

range_rules = {
    "unemployment_pct": (0, 100),
    "bank_npl_pct_gross_loans": (0, 100),
    "rating_score_mean_t": (0, 100),
    "rating_score_mean_t1": (0, 100)
}


range_checks = []


for variable, limits in range_rules.items():

    if variable not in clean_panel.columns:
        continue

    lower_limit, upper_limit = limits

    invalid_values = (
        clean_panel[variable].notna()
        &
        (
            (clean_panel[variable] < lower_limit)
            |
            (clean_panel[variable] > upper_limit)
        )
    )

    range_checks.append({
        "variable": variable,
        "lower_limit": lower_limit,
        "upper_limit": upper_limit,
        "invalid_values": invalid_values.sum()
    })


range_checks = pd.DataFrame(range_checks)


display(range_checks)

,variable,lower_limit,upper_limit,invalid_values
0,unemployment_pct,0,100,0
1,bank_npl_pct_gross_loans,0,100,0
2,rating_score_mean_t,0,100,0
3,rating_score_mean_t1,0,100,0


In [57]:
# Variables que, por definición, no pueden ser negativas

non_negative_variables = [
    "gdp_per_capita_constant_2015_usd",
    "external_debt_stocks_current_usd",
    "external_debt_pct_gni",
    "external_debt_service_pct_exports",
    "short_term_debt_pct_reserves",
    "reserves_pct_external_debt",
    "reserves_months_imports",
    "government_debt_pct_gdp",
    "bank_npl_pct_gross_loans",
    "head_of_government_tenure_years",
    "average_government_duration_since_2000",
    "government_changes_previous_5y",
    "battle_related_deaths",
    "log_battle_related_deaths"
]


non_negative_variables = [
    variable
    for variable in non_negative_variables
    if variable in clean_panel.columns
]


negative_checks = pd.DataFrame({
    "variable": non_negative_variables,

    "negative_values": [
        (
            clean_panel[variable] < 0
        ).sum()
        for variable in non_negative_variables
    ],

    "minimum": [
        clean_panel[variable].min()
        for variable in non_negative_variables
    ]
})


display(negative_checks)

,variable,negative_values,minimum
0,gdp_per_capita_constant_2015_usd,0,233.0324
1,external_debt_stocks_current_usd,0,"69,583,145.7000"
2,external_debt_pct_gni,0,1.1542
3,external_debt_service_pct_exports,0,0.0035
4,short_term_debt_pct_reserves,0,0.0000
5,reserves_pct_external_debt,0,0.0094
6,reserves_months_imports,0,0.0100
7,government_debt_pct_gdp,0,0.0000
8,bank_npl_pct_gross_loans,0,0.0923
9,head_of_government_tenure_years,0,1.0000


## 7. División temporal de los datos

La base se divide respetando el orden temporal:

- Entrenamiento: 2000-2018.
- Validación: 2019-2021.
- Prueba: 2022-2024.

Los años se refieren a las variables explicativas. Por tanto, el conjunto de prueba utiliza información de 2022-2024 para predecir los ratings de 2023-2025.

La imputación, el escalado y el tratamiento de valores extremos se ajustarán posteriormente utilizando únicamente el conjunto de entrenamiento.

In [58]:
# Definir el año que se quiere predecir

clean_panel["target_year"] = (
    clean_panel["year"] + 1
)


# Comprobar que coincide con el año del rating disponible

rows_with_target = (
    clean_panel["target_available_t1"] == 1
)

incorrect_target_year = (
    clean_panel.loc[
        rows_with_target,
        "target_year"
    ]
    !=
    clean_panel.loc[
        rows_with_target,
        "rating_year"
    ]
).sum()


print(
    "Años objetivo incorrectos:",
    incorrect_target_year
)

Años objetivo incorrectos: 0


In [59]:
# Crear la división temporal

clean_panel["data_split"] = pd.NA


clean_panel.loc[
    clean_panel["year"] <= 2018,
    "data_split"
] = "train"


clean_panel.loc[
    clean_panel["year"].between(
        2019,
        2021
    ),
    "data_split"
] = "validation"


clean_panel.loc[
    clean_panel["year"].between(
        2022,
        2024
    ),
    "data_split"
] = "test"

In [60]:
# Conservar solo las filas con objetivo disponible

model_sample = clean_panel[
    clean_panel["target_available_t1"] == 1
].copy()


print(
    "Filas de la base completa:",
    len(clean_panel)
)

print(
    "Filas de la muestra del modelo:",
    len(model_sample)
)

Filas de la base completa: 5425
Filas de la muestra del modelo: 3434


In [61]:
# Resumir cada periodo

split_summary = (
    model_sample
    .groupby(
        "data_split"
    )
    .agg(
        rows=(
            "iso3",
            "size"
        ),
        entities=(
            "iso3",
            "nunique"
        ),
        first_predictor_year=(
            "year",
            "min"
        ),
        last_predictor_year=(
            "year",
            "max"
        ),
        first_target_year=(
            "target_year",
            "min"
        ),
        last_target_year=(
            "target_year",
            "max"
        )
    )
    .reset_index()
)


split_order = [
    "train",
    "validation",
    "test"
]


split_summary["data_split"] = pd.Categorical(
    split_summary["data_split"],
    categories=split_order,
    ordered=True
)


split_summary = split_summary.sort_values(
    "data_split"
).reset_index(
    drop=True
)


display(split_summary)

,data_split,rows,entities,first_predictor_year,last_predictor_year,first_target_year,last_target_year
0,train,2497,155,2000,2018,2001,2019
1,validation,466,156,2019,2021,2020,2022
2,test,471,158,2022,2024,2023,2025


In [62]:
# Comprobar los años de cada conjunto

print(
    "Años de entrenamiento:",
    sorted(
        model_sample.loc[
            model_sample["data_split"] == "train",
            "year"
        ].unique()
    )
)

print(
    "Años de validación:",
    sorted(
        model_sample.loc[
            model_sample["data_split"] == "validation",
            "year"
        ].unique()
    )
)

print(
    "Años de prueba:",
    sorted(
        model_sample.loc[
            model_sample["data_split"] == "test",
            "year"
        ].unique()
    )
)

Años de entrenamiento: [np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018)]
Años de validación: [np.int64(2019), np.int64(2020), np.int64(2021)]
Años de prueba: [np.int64(2022), np.int64(2023), np.int64(2024)]


In [63]:
# Comprobar los años de cada conjunto

print(
    "Años de entrenamiento:",
    sorted(
        model_sample.loc[
            model_sample["data_split"] == "train",
            "year"
        ].unique()
    )
)

print(
    "Años de validación:",
    sorted(
        model_sample.loc[
            model_sample["data_split"] == "validation",
            "year"
        ].unique()
    )
)

print(
    "Años de prueba:",
    sorted(
        model_sample.loc[
            model_sample["data_split"] == "test",
            "year"
        ].unique()
    )
)

Años de entrenamiento: [np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018)]
Años de validación: [np.int64(2019), np.int64(2020), np.int64(2021)]
Años de prueba: [np.int64(2022), np.int64(2023), np.int64(2024)]


In [65]:
# Guardar la base preparada, todavía sin imputar ni transformar

PREPARED_PANEL_FILE = (
    MODEL_DATA_PATH
    / "country_risk_panel_prepared_2000_2024.csv"
)

clean_panel.to_csv(
    PREPARED_PANEL_FILE,
    index=False,
    encoding="utf-8-sig"
)


# Guardar la muestra con objetivo disponible

MODEL_SAMPLE_FILE = (
    MODEL_DATA_PATH
    / "country_risk_model_sample_raw_2000_2024.csv"
)

model_sample.to_csv(
    MODEL_SAMPLE_FILE,
    index=False,
    encoding="utf-8-sig"
)


print(
    "Base preparada guardada en:",
    PREPARED_PANEL_FILE
)

print(
    "Muestra sin transformar guardada en:",
    MODEL_SAMPLE_FILE
)

Base preparada guardada en: data\model\country_risk_panel_prepared_2000_2024.csv
Muestra sin transformar guardada en: data\model\country_risk_model_sample_raw_2000_2024.csv


## Resumen de la preparación realizada

Hasta este punto se ha construido una única base de datos integrada para la modelización del rating soberano.

En primer lugar, se han unido los distintos bloques de variables económicas, fiscales, externas, monetarias, institucionales, políticas, de conflicto y de historial de default mediante las claves `iso3` y `year`.

La base final contiene 217 entidades durante el periodo 2000-2024, con un total de 5.425 observaciones entidad-año.

Posteriormente, se ha incorporado como variable objetivo el rating soberano medio del año siguiente. De esta forma, las variables explicativas del año \(t\) se utilizan para predecir el rating correspondiente al año \(t+1\). La alineación temporal ha sido comprobada y no existen relaciones incorrectas ni observaciones duplicadas.

La muestra dispone de 3.434 observaciones con rating futuro disponible, lo que representa el 63,3 % del panel integrado.

También se ha incorporado el rating observado en el año \(t\), que permitirá comparar un modelo basado únicamente en variables fundamentales con otro que incluya además la información del rating actual, así como construir un modelo baseline de persistencia.

A continuación se ha realizado una auditoría general de las variables. Se ha estudiado su tipo, cobertura, porcentaje de valores ausentes y disponibilidad temporal. No se han detectado variables constantes ni valores infinitos.

Asimismo, se han revisado los rangos de las variables binarias y de aquellas variables con límites económicos claros. No se han encontrado valores imposibles: las variables binarias toman únicamente valores 0 y 1, los ratings se encuentran entre 0 y 100 y las variables que por definición deben ser no negativas no presentan valores inferiores a cero.

Por tanto, la estructura de la base, la relación temporal entre predictores y objetivo y la consistencia básica de los datos ya han sido verificadas. La imputación, el tratamiento de valores extremos, las transformaciones y el escalado todavía no se han aplicado, ya que dependerán de las necesidades de cada grupo de modelos.

In [66]:
# Variables que nunca entran como predictores

columns_to_exclude = [
    "iso3",
    "iso2",
    "country",
    "year",
    "economic_period",
    "head_of_government",
    "rating_year",
    "target_year",
    "target_available_t1",
    "data_split",
    "rating_change_t1",
    "rating_available_t",
    "persistence_available",
    "number_of_ratings_t",
    "has_at_least_one_rating_t"
]


# Añadir todas las variables futuras t+1

columns_to_exclude += [
    column
    for column in clean_panel.columns
    if column.endswith("_t1")
]


# Variables con más del 40 % de ausentes

high_missing_variables = variable_audit.loc[
    variable_audit["missing_model_sample_pct"] > 40,
    "variable"
].tolist()


columns_to_exclude += high_missing_variables


# El rating actual se reserva para el segundo modelo

columns_to_exclude += [
    "rating_score_mean_t"
]


columns_to_exclude = list(
    set(columns_to_exclude)
)


# Variables del modelo principal

model_variables = [
    column
    for column in clean_panel.columns
    if column not in columns_to_exclude
]


print(
    "Variables del modelo principal:",
    len(model_variables)
)

print(
    "\nVariables utilizadas:"
)

print(
    model_variables
)

Variables del modelo principal: 43

Variables utilizadas:
['region', 'income_level', 'exchange_rate_lcu_per_usd', 'gdp_growth_pct', 'gdp_per_capita_constant_2015_usd', 'gdp_per_capita_ppp_constant_2021_intl_usd', 'gdp_ppp_constant_2021_intl_usd', 'gross_savings_pct_gdp', 'inflation_cpi_pct', 'investment_pct_gdp', 'unemployment_pct', 'fiscal_balance_pct_gdp', 'government_debt_pct_gdp', 'government_expenditure_pct_gdp', 'government_revenue_pct_gdp', 'primary_balance_pct_gdp', 'net_interest_payments_pct_gdp', 'current_account_balance_pct_gdp', 'exports_growth_pct', 'exports_pct_gdp', 'fdi_net_inflows_pct_gdp', 'imports_pct_gdp', 'reserves_months_imports', 'trade_openness_pct_gdp', 'real_interest_rate_pct', 'control_of_corruption_estimate', 'government_effectiveness_estimate', 'political_stability_estimate', 'regulatory_quality_estimate', 'rule_of_law_estimate', 'voice_accountability_estimate', 'electoral_democracy_index', 'head_of_government_tenure_years', 'average_government_duration_sin

In [67]:
future_variables = [
    column
    for column in model_variables
    if column.endswith("_t1")
]


print(
    "Variables futuras dentro de X:",
    future_variables
)

Variables futuras dentro de X: []


In [70]:
# Modelo 1: variables fundamentales

model_variables_fundamentals = model_variables.copy()

# Modelo 2: variables fundamentales + rating actual

model_variables_rating = (
    model_variables
    +
    ["rating_score_mean_t"]
)


print(
    "Modelo fundamentales:",
    len(model_variables_fundamentals),
    "variables"
)

print(
    "Modelo fundamentales + rating:",
    len(model_variables_rating),
    "variables"
)

Modelo fundamentales: 43 variables
Modelo fundamentales + rating: 44 variables


Debido a la mala cobertura, es decir > 40% de missings se han eliminado las siguientes variables:

- `external_debt_pct_gni`
- `external_debt_service_pct_exports`
- `external_debt_stocks_current_usd`
- `reserves_pct_external_debt`
- `short_term_debt_pct_reserves`
- `bank_npl_pct_gross_loans`
- `global_competitiveness_index`
- `battle_related_deaths`
- `armed_conflict_dummy`
- `log_battle_related_deaths`

y `head_of_government` porque es el nombre de una persona con cientos de categorias y no tiene sentido que el modelo aprenda por ejemplo, “Pedro Sánchez” o “Angela Merkel” como si fueran una característica cuantitativa del país. La auditoría del notebook confirma precisamente la mala cobertura de deuda externa, conflicto, competitividad, etc.

### Variables predictoras seleccionadas

| Grupo | Variable |
|---|---|
| **Macroeconómicas** | `exchange_rate_lcu_per_usd` |
|  | `gdp_growth_pct` |
|  | `gdp_per_capita_constant_2015_usd` |
|  | `gdp_per_capita_ppp_constant_2021_intl_usd` |
|  | `gdp_ppp_constant_2021_intl_usd` |
|  | `gross_savings_pct_gdp` |
|  | `inflation_cpi_pct` |
|  | `investment_pct_gdp` |
|  | `unemployment_pct` |
| **Fiscales** | `fiscal_balance_pct_gdp` |
|  | `government_debt_pct_gdp` |
|  | `government_expenditure_pct_gdp` |
|  | `government_revenue_pct_gdp` |
|  | `primary_balance_pct_gdp` |
|  | `net_interest_payments_pct_gdp` |
| **Sector exterior** | `current_account_balance_pct_gdp` |
|  | `exports_growth_pct` |
|  | `exports_pct_gdp` |
|  | `fdi_net_inflows_pct_gdp` |
|  | `imports_pct_gdp` |
|  | `reserves_months_imports` |
|  | `trade_openness_pct_gdp` |
| **Financieras** | `real_interest_rate_pct` |
| **Institucionales** | `control_of_corruption_estimate` |
|  | `government_effectiveness_estimate` |
|  | `political_stability_estimate` |
|  | `regulatory_quality_estimate` |
|  | `rule_of_law_estimate` |
|  | `voice_accountability_estimate` |
| **Democracia** | `electoral_democracy_index` |
| **Políticas** | `head_of_government_tenure_years` |
|  | `average_government_duration_since_2000` |
|  | `government_change_dummy` |
|  | `government_changes_previous_5y` |
|  | `fragile_states_index` |
| **Historial de default soberano** | `sovereign_debt_in_default_usd_mn` |
|  | `sovereign_default_dummy` |
|  | `default_previous_5y` |
|  | `default_previous_10y` |
|  | `years_since_last_default` |
|  | `never_default_history_dummy` |

**Total: 41 variables predictoras fundamentales.**

Además, se conservan `region` e `income_level` como variables categóricas, por lo que el modelo de fundamentales dispone de **43 variables de entrada**.

El segundo planteamiento incorpora adicionalmente `rating_score_mean_t`, correspondiente al rating observado en el año actual, alcanzando **44 variables de entrada**.

In [71]:
9+6+7+1+6+1+5+6

41

A lo que se añadiria `region` e `income_level`

In [72]:
print("Número de predictoras:", len(model_variables_fundamentals))

for variable in model_variables_fundamentals:
    print(variable)

Número de predictoras: 43
region
income_level
exchange_rate_lcu_per_usd
gdp_growth_pct
gdp_per_capita_constant_2015_usd
gdp_per_capita_ppp_constant_2021_intl_usd
gdp_ppp_constant_2021_intl_usd
gross_savings_pct_gdp
inflation_cpi_pct
investment_pct_gdp
unemployment_pct
fiscal_balance_pct_gdp
government_debt_pct_gdp
government_expenditure_pct_gdp
government_revenue_pct_gdp
primary_balance_pct_gdp
net_interest_payments_pct_gdp
current_account_balance_pct_gdp
exports_growth_pct
exports_pct_gdp
fdi_net_inflows_pct_gdp
imports_pct_gdp
reserves_months_imports
trade_openness_pct_gdp
real_interest_rate_pct
control_of_corruption_estimate
government_effectiveness_estimate
political_stability_estimate
regulatory_quality_estimate
rule_of_law_estimate
voice_accountability_estimate
electoral_democracy_index
head_of_government_tenure_years
average_government_duration_since_2000
government_change_dummy
government_changes_previous_5y
fragile_states_index
sovereign_debt_in_default_usd_mn
sovereign_defaul

## 8. Creación de variables derivadas *(feature engineering)*

Antes de comenzar el preprocesamiento específico de los modelos, se crean variables derivadas que pueden aportar información adicional sobre la situación y evolución económica de cada país.

Estas variables se construyen utilizando únicamente información disponible en el año actual y en años anteriores, evitando utilizar información futura.

Además de las variables originales, se incorporan variables derivadas con el objetivo de representar la evolución reciente, la persistencia, la volatilidad y determinadas situaciones de vulnerabilidad económica.

Todas las variables temporales se construyen utilizando únicamente información disponible en el año \(t\) y en años anteriores, evitando incorporar información futura.

| Bloque | Variable derivada | Descripción |
|---|---|---|
| **Macroeconómico** | `gdp_growth_3y_mean` | Crecimiento medio del PIB durante los últimos 3 años |
|  | `gdp_growth_3y_std` | Volatilidad del crecimiento del PIB durante los últimos 3 años |
|  | `recession_dummy` | Indicador de crecimiento negativo del PIB |
|  | `log_gdp_per_capita` | Logaritmo del PIB real per cápita |
|  | `inflation_abs` | Valor absoluto de la inflación |
|  | `high_inflation_dummy` | Indicador de inflación superior al 10 % |
|  | `inflation_3y_std` | Volatilidad de la inflación durante los últimos 3 años |
|  | `unemployment_change` | Cambio anual de la tasa de desempleo |
|  | `saving_investment_gap` | Diferencia entre ahorro e inversión sobre el PIB |
|  | `exchange_rate_log_change` | Variación logarítmica anual del tipo de cambio |
|  | `log_gdp_ppp` | Logaritmo del PIB ajustado por PPA |
|  | `log_gdp_per_capita_ppp` | Logaritmo del PIB per cápita ajustado por PPA |
| **Fiscal** | `government_debt_change` | Cambio anual de la deuda pública sobre el PIB |
|  | `government_debt_3y_change` | Cambio de la deuda pública respecto a tres años antes |
|  | `fiscal_balance_3y_mean` | Saldo fiscal medio durante los últimos 3 años |
|  | `fiscal_deficit_dummy` | Indicador de déficit fiscal |
|  | `persistent_deficit_dummy` | Indicador de déficit fiscal persistente |
|  | `primary_balance_3y_mean` | Saldo primario medio durante los últimos 3 años |
|  | `primary_deficit_dummy` | Indicador de déficit primario |
|  | `government_revenue_change` | Cambio anual de los ingresos públicos sobre el PIB |
|  | `government_revenue_3y_mean` | Ingresos públicos medios durante los últimos 3 años |
|  | `government_expenditure_change` | Cambio anual del gasto público sobre el PIB |
|  | `government_expenditure_3y_mean` | Gasto público medio durante los últimos 3 años |
| **Sector exterior** | `current_account_3y_mean` | Saldo por cuenta corriente medio durante los últimos 3 años |
|  | `current_account_deficit_dummy` | Indicador de déficit por cuenta corriente |
|  | `persistent_current_account_deficit` | Indicador de déficit exterior persistente |
|  | `exports_growth_3y_mean` | Crecimiento medio de las exportaciones durante los últimos 3 años |
|  | `exports_growth_3y_std` | Volatilidad del crecimiento de las exportaciones durante los últimos 3 años |
|  | `exports_contraction_dummy` | Indicador de contracción de las exportaciones |
| **Monetario-financiero** | `real_interest_rate_5y_mean` | Tipo de interés real medio durante los últimos 5 años |
|  | `real_interest_rate_3y_std` | Volatilidad del tipo de interés real durante los últimos 3 años |
|  | `real_interest_rate_change` | Cambio anual del tipo de interés real |
|  | `negative_real_interest_dummy` | Indicador de tipo de interés real negativo |

**Total: 33 variables derivadas propuestas.**

Las variables derivadas asociadas a indicadores que han sido excluidos del modelo principal por su baja cobertura, como deuda externa o préstamos bancarios no productivos, no se incorporan en esta especificación principal y podrán analizarse posteriormente en una especificación ampliada.

Ahora — feature engineering común, usando únicamente t y pasado:

    medias móviles;
    desviaciones móviles;
    cambios anuales;
    cambios a 3 años;
    dummies;
    saving_investment_gap;
    persistencia de déficits.

Después del split — transformaciones:

    log_gdp_per_capita;
    log_gdp_ppp;
    log_gdp_per_capita_ppp;
    transformaciones de variables muy asimétricas;
    winsorización;
    imputación;
    escalado.


Por lo que quedan 29 variables derivadas comunes que si se crearán ahora.

    29 variables derivadas
            ↓
    split temporal
            ↓            
    Rama A / Rama B -> (transformaciones logaritmicas)

    BASE INTEGRADA
          ↓
    FEATURE ENGINEERING COMÚN
    - medias 3y / 5y
    - desviaciones 3y
    - cambios anuales
    - cambios a 3 años
    - dummies
    - saving_investment_gap
    - exchange_rate_log_change
          ↓
    SPLIT TEMPORAL
    Train / Validation / Test
          ↓
    ┌────────────────────────┬────────────────────────┐
    │        RAMA A          │        RAMA B          │
    │ modelos sensibles      │ árboles                │
    │                        │                        │
    │ imputación             │ missing según modelo   │
    │ logaritmos             │ normalmente sin log    │
    │ outliers/winsorización │ sin escalado           │
    │ escalado               │                        │
    │ PCA si procede         │                        │
    └────────────────────────┴────────────────────────┘

In [74]:
# ----------------------------------
# FEATURE ENGINEERING
# ----------------------------------

# Ordenar por país y año

clean_panel = clean_panel.sort_values(
    ["iso3", "year"]
).reset_index(drop=True)


# 1. VARIABLES MACROECONÓMICAS

clean_panel["gdp_growth_3y_mean"] = (
    clean_panel
    .groupby("iso3")["gdp_growth_pct"]
    .transform(
        lambda x: x.rolling(3, min_periods=3).mean()
    )
)

clean_panel["gdp_growth_3y_std"] = (
    clean_panel
    .groupby("iso3")["gdp_growth_pct"]
    .transform(
        lambda x: x.rolling(3, min_periods=3).std()
    )
)

clean_panel["recession_dummy"] = np.where(
    clean_panel["gdp_growth_pct"].isna(),
    np.nan,
    (clean_panel["gdp_growth_pct"] < 0).astype(int)
)

clean_panel["inflation_abs"] = (
    clean_panel["inflation_cpi_pct"].abs()
)

clean_panel["high_inflation_dummy"] = np.where(
    clean_panel["inflation_cpi_pct"].isna(),
    np.nan,
    (clean_panel["inflation_cpi_pct"] > 10).astype(int)
)

clean_panel["inflation_3y_std"] = (
    clean_panel
    .groupby("iso3")["inflation_cpi_pct"]
    .transform(
        lambda x: x.rolling(3, min_periods=3).std()
    )
)

clean_panel["unemployment_change"] = (
    clean_panel
    .groupby("iso3")["unemployment_pct"]
    .diff()
)

clean_panel["saving_investment_gap"] = (
    clean_panel["gross_savings_pct_gdp"]
    - clean_panel["investment_pct_gdp"]
)


# 2. VARIABLES FISCALES

clean_panel["government_debt_change"] = (
    clean_panel
    .groupby("iso3")["government_debt_pct_gdp"]
    .diff()
)

clean_panel["government_debt_3y_change"] = (
    clean_panel
    .groupby("iso3")["government_debt_pct_gdp"]
    .diff(3)
)

clean_panel["fiscal_balance_3y_mean"] = (
    clean_panel
    .groupby("iso3")["fiscal_balance_pct_gdp"]
    .transform(
        lambda x: x.rolling(3, min_periods=3).mean()
    )
)

clean_panel["fiscal_deficit_dummy"] = np.where(
    clean_panel["fiscal_balance_pct_gdp"].isna(),
    np.nan,
    (clean_panel["fiscal_balance_pct_gdp"] < 0).astype(int)
)

clean_panel["persistent_deficit_dummy"] = (
    clean_panel
    .groupby("iso3")["fiscal_balance_pct_gdp"]
    .transform(
        lambda x: x.rolling(
            3,
            min_periods=3
        ).apply(
            lambda y: int((y < 0).all()),
            raw=True
        )
    )
)

clean_panel["primary_balance_3y_mean"] = (
    clean_panel
    .groupby("iso3")["primary_balance_pct_gdp"]
    .transform(
        lambda x: x.rolling(3, min_periods=3).mean()
    )
)

clean_panel["primary_deficit_dummy"] = np.where(
    clean_panel["primary_balance_pct_gdp"].isna(),
    np.nan,
    (clean_panel["primary_balance_pct_gdp"] < 0).astype(int)
)

clean_panel["government_revenue_change"] = (
    clean_panel
    .groupby("iso3")["government_revenue_pct_gdp"]
    .diff()
)

clean_panel["government_revenue_3y_mean"] = (
    clean_panel
    .groupby("iso3")["government_revenue_pct_gdp"]
    .transform(
        lambda x: x.rolling(3, min_periods=3).mean()
    )
)

clean_panel["government_expenditure_change"] = (
    clean_panel
    .groupby("iso3")["government_expenditure_pct_gdp"]
    .diff()
)

clean_panel["government_expenditure_3y_mean"] = (
    clean_panel
    .groupby("iso3")["government_expenditure_pct_gdp"]
    .transform(
        lambda x: x.rolling(3, min_periods=3).mean()
    )
)


# 3. VARIABLES DEL SECTOR EXTERIOR

clean_panel["current_account_3y_mean"] = (
    clean_panel
    .groupby("iso3")["current_account_balance_pct_gdp"]
    .transform(
        lambda x: x.rolling(3, min_periods=3).mean()
    )
)

clean_panel["current_account_deficit_dummy"] = np.where(
    clean_panel["current_account_balance_pct_gdp"].isna(),
    np.nan,
    (
        clean_panel["current_account_balance_pct_gdp"] < 0
    ).astype(int)
)

clean_panel["persistent_current_account_deficit"] = (
    clean_panel
    .groupby("iso3")["current_account_balance_pct_gdp"]
    .transform(
        lambda x: x.rolling(
            3,
            min_periods=3
        ).apply(
            lambda y: int((y < 0).all()),
            raw=True
        )
    )
)

clean_panel["exports_growth_3y_mean"] = (
    clean_panel
    .groupby("iso3")["exports_growth_pct"]
    .transform(
        lambda x: x.rolling(3, min_periods=3).mean()
    )
)

clean_panel["exports_growth_3y_std"] = (
    clean_panel
    .groupby("iso3")["exports_growth_pct"]
    .transform(
        lambda x: x.rolling(3, min_periods=3).std()
    )
)

clean_panel["exports_contraction_dummy"] = np.where(
    clean_panel["exports_growth_pct"].isna(),
    np.nan,
    (clean_panel["exports_growth_pct"] < 0).astype(int)
)


# 4. VARIABLES MONETARIAS Y FINANCIERAS

clean_panel["real_interest_rate_5y_mean"] = (
    clean_panel
    .groupby("iso3")["real_interest_rate_pct"]
    .transform(
        lambda x: x.rolling(5, min_periods=5).mean()
    )
)

clean_panel["real_interest_rate_3y_std"] = (
    clean_panel
    .groupby("iso3")["real_interest_rate_pct"]
    .transform(
        lambda x: x.rolling(3, min_periods=3).std()
    )
)

clean_panel["real_interest_rate_change"] = (
    clean_panel
    .groupby("iso3")["real_interest_rate_pct"]
    .diff()
)

clean_panel["negative_real_interest_dummy"] = np.where(
    clean_panel["real_interest_rate_pct"].isna(),
    np.nan,
    (clean_panel["real_interest_rate_pct"] < 0).astype(int)
)

In [75]:
# Variables derivadas

derived_variables = [
    "gdp_growth_3y_mean",
    "gdp_growth_3y_std",
    "recession_dummy",
    "inflation_abs",
    "high_inflation_dummy",
    "inflation_3y_std",
    "unemployment_change",
    "saving_investment_gap",

    "government_debt_change",
    "government_debt_3y_change",
    "fiscal_balance_3y_mean",
    "fiscal_deficit_dummy",
    "persistent_deficit_dummy",
    "primary_balance_3y_mean",
    "primary_deficit_dummy",
    "government_revenue_change",
    "government_revenue_3y_mean",
    "government_expenditure_change",
    "government_expenditure_3y_mean",

    "current_account_3y_mean",
    "current_account_deficit_dummy",
    "persistent_current_account_deficit",
    "exports_growth_3y_mean",
    "exports_growth_3y_std",
    "exports_contraction_dummy",

    "real_interest_rate_5y_mean",
    "real_interest_rate_3y_std",
    "real_interest_rate_change",
    "negative_real_interest_dummy"
]


print(
    "Variables derivadas creadas:",
    len(derived_variables)
)

Variables derivadas creadas: 29


In [76]:
# Variables fundamentales originales válidas

base_variables = [
    variable
    for variable in model_variables
    if variable != "exchange_rate_lcu_per_usd"
]


# Añadir las variables derivadas

model_variables_fundamentals = (
    base_variables
    +
    derived_variables
)


# Segundo modelo: añadir rating actual

model_variables_rating = (
    model_variables_fundamentals
    +
    ["rating_score_mean_t"]
)


print(
    "Variables fundamentales:",
    len(model_variables_fundamentals)
)

print(
    "Variables fundamentales + rating:",
    len(model_variables_rating)
)

Variables fundamentales: 71
Variables fundamentales + rating: 72


In [78]:
# Actualizar la muestra modelizable

model_sample = clean_panel[
    clean_panel["target_available_t1"] == 1
].copy()

## División temporal de la muestra

Una vez integrada la información, seleccionado el conjunto de variables explicativas y creadas las variables derivadas comunes, se realiza la división temporal de la muestra.

Debido a que el objetivo del trabajo consiste en predecir la calificación crediticia futura, no se realiza una partición aleatoria de las observaciones. Una división aleatoria permitiría que observaciones de años posteriores participasen en el entrenamiento de modelos utilizados para predecir años anteriores, generando una situación poco realista y pudiendo producir fuga de información.

La muestra se divide respetando el orden cronológico:

- **Entrenamiento:** 2000-2018.
- **Validación:** 2019-2021.
- **Test:** 2022-2024.

Los años anteriores corresponden al año de las variables explicativas. Como el objetivo se encuentra desplazado un año hacia delante, los periodos que realmente se predicen son:

| Conjunto | Años de las variables \(X_t\) | Años del rating objetivo \(Rating_{t+1}\) |
|---|---:|---:|
| Entrenamiento | 2000-2018 | 2001-2019 |
| Validación | 2019-2021 | 2020-2022 |
| Test | 2022-2024 | 2023-2025 |

El conjunto de entrenamiento se utilizará para ajustar los modelos y todos los procedimientos que deban aprender parámetros a partir de los datos, como la imputación, el escalado, el tratamiento de valores extremos o determinadas transformaciones.

El conjunto de validación se utilizará para seleccionar modelos e hiperparámetros, mientras que el conjunto de test permanecerá sin utilizar durante el desarrollo y se empleará únicamente para realizar la evaluación final.

Esta misma división temporal se mantendrá para todos los modelos, permitiendo comparar sus resultados sobre exactamente los mismos periodos.

A partir de esta base se desarrollarán dos ramas de preprocesamiento:

- **Rama A:** modelos sensibles a la escala y a la distribución de las variables.
- **Rama B:** modelos basados en árboles.

Cada rama aplicará posteriormente su propio preprocesamiento utilizando exclusivamente la información del conjunto de entrenamiento.

In [79]:
# Crear de nuevo la muestra con todas las variables derivadas

model_sample = clean_panel[
    clean_panel["target_available_t1"] == 1
].copy()


# Asignar la división temporal

model_sample["data_split"] = pd.NA

model_sample.loc[
    model_sample["year"] <= 2018,
    "data_split"
] = "train"

model_sample.loc[
    model_sample["year"].between(2019, 2021),
    "data_split"
] = "validation"

model_sample.loc[
    model_sample["year"].between(2022, 2024),
    "data_split"
] = "test"


# Comprobar la división

split_summary = (
    model_sample
    .groupby("data_split")
    .agg(
        observations=("iso3", "size"),
        countries=("iso3", "nunique"),
        first_year=("year", "min"),
        last_year=("year", "max")
    )
)

display(split_summary)

,observations,countries,first_year,last_year
data_split,,,,
test,471,158,2022,2024
train,2497,155,2000,2018
validation,466,156,2019,2021


In [80]:
# Guardar la base preparada para modelización

MODEL_READY_FILE = (
    MODEL_DATA_PATH
    / "country_risk_model_ready_2000_2024.csv"
)

model_sample.to_csv(
    MODEL_READY_FILE,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Base preparada para modelización:",
    MODEL_READY_FILE
)

Base preparada para modelización: data\model\country_risk_model_ready_2000_2024.csv


## Base final para modelización

Tras la integración, depuración y preparación de los distintos bloques de información, se obtiene una base país-año en la que las variables observadas en el año \(t\) se utilizan para predecir el rating soberano del año \(t+1\).

La base original integrada comprende el periodo 2000-2024 y parte de 5.425 observaciones país-año. Para la modelización se conservan las 3.434 observaciones para las que existe un rating objetivo en el año siguiente.

Se han eliminado como predictores los identificadores, las variables con una cobertura insuficiente y aquellas que podrían producir fuga de información. Asimismo, se han incorporado variables derivadas que recogen cambios, tendencias, volatilidad, persistencia y situaciones de vulnerabilidad económica utilizando exclusivamente información contemporánea y pasada.

Finalmente, la muestra se ha dividido temporalmente en entrenamiento (2000-2018), validación (2019-2021) y test (2022-2024).

La base todavía no ha sido imputada, escalada ni sometida a transformaciones estadísticas. Estas operaciones se realizarán en el Notebook 4 de forma independiente para cada familia de modelos:

- **Rama A:** imputación, transformaciones logarítmicas cuando proceda, tratamiento de valores extremos, codificación de variables categóricas y escalado.
- **Rama B:** tratamiento de valores ausentes y variables categóricas según las necesidades de los modelos basados en árboles, generalmente sin necesidad de escalado ni de transformaciones logarítmicas por asimetría.

Todos los procedimientos que requieran estimar parámetros a partir de los datos se ajustarán exclusivamente utilizando el conjunto de entrenamiento.